---

# Unit 3 - Assignment: Building a Production Advanced RAG System

## Objective

Build a full **Advanced RAG pipeline** that goes significantly beyond Naïve RAG. You will combine every retrieval technique from this unit into a single working system and demonstrate that each step measurably improves result quality.

**Context**: You are building an internal knowledge assistant for a university. The system must answer student questions about AI/ML topics using a provided document corpus. The challenge: student questions are short and vague (*"what is attention?"*), while the documents use precise technical vocabulary. Your pipeline must handle this vocabulary gap reliably.

### Pipeline Overview

```mermaid
graph LR
    U["User Query"] --> QE["Query Expansion (HyDE / Multi-Query)"]
    QE --> HR["Hybrid Retrieval (BM25 + SBERT + RRF)"]
    HR --> RR["Cross-Encoder Re-Ranking"]
    RR --> LLM["LLM Generation (Groq (Llama))"]
    LLM --> A["Answer"]
```

In [1]:
%pip install python-dotenv rank-bm25 sentence-transformers langchain langchain-google-genai langchain-community langchain-huggingface numpy --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\diyab\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os, getpass

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

print("Setup complete.")

Setup complete.


---

## Part 1 - Document Corpus Setup

We construct a corpus of **15 AI/ML documents** deliberately designed to stress-test our retrieval system:

- At least 3 documents cover related but distinct sub-topics within neural network training.
- At least 1 document contains a technical proper noun (`BM25`, `AdamW`, `GPTQ`) that BM25 will find precisely but dense embeddings may miss.
- Documents vary in vocabulary to expose weaknesses in both sparse and dense retrieval.

In [3]:
import numpy as np

corpus = [
    # doc_0 — Transformers / Attention
    "Transformers use self-attention mechanisms to process sequences in parallel, replacing recurrent networks entirely.",
    # doc_1 — BERT
    "BERT is a bidirectional encoder pre-trained using masked language modelling and next sentence prediction.",
    # doc_2 — BM25 (technical proper noun — BM25 will excel here)
    "The BM25 algorithm ranks documents using term frequency saturation and inverse document frequency with length normalization.",
    # doc_3 — Gradient Descent
    "Gradient descent minimizes the loss function by iteratively updating weights in the direction of the negative gradient.",
    # doc_4 — Backpropagation (neural network training — angle 1)
    "Neural networks learn by propagating the error signal backwards through layers using the chain rule of calculus.",
    # doc_5 — Weight Initialization (neural network training — angle 2)
    "Proper weight initialization prevents vanishing and exploding gradients, enabling stable training of deep networks.",
    # doc_6 — Batch Normalization (neural network training — angle 3)
    "Batch normalization normalizes layer inputs during training, accelerating convergence and acting as a regularizer.",
    # doc_7 — RAG
    "Retrieval Augmented Generation grounds language model responses by injecting retrieved documents into the prompt context.",
    # doc_8 — LoRA / Fine-Tuning
    "LoRA fine-tunes large language models by injecting trainable low-rank matrices into frozen attention weight projections.",
    # doc_9 — Quantization (proper noun: GPTQ, INT8)
    "GPTQ quantizes model weights to INT4 post-training by minimizing layer-wise reconstruction error on a calibration set.",
    # doc_10 — Mixture of Experts
    "Mixture of Experts models activate only a sparse subset of expert networks per token, controlled by a learned router.",
    # doc_11 — AdamW Optimizer (proper noun: AdamW)
    "AdamW decouples weight decay from the gradient update in the Adam optimizer, improving regularization in transformer training.",
    # doc_12 — Cosine Similarity
    "Cosine similarity measures the angle between two vectors and is used to compare query and document embeddings in dense retrieval.",
    # doc_13 — Dropout
    "Dropout randomly deactivates neurons during training to prevent co-adaptation and improve generalization of deep models.",
    # doc_14 — Cross-Entropy Loss
    "Cross-entropy loss measures the divergence between predicted probability distributions and ground-truth labels during classification.",
]

print(f"Corpus size: {len(corpus)} documents")
print()
for i, doc in enumerate(corpus):
    print(f"  doc_{i:02d}: {doc[:80]}")

Corpus size: 15 documents

  doc_00: Transformers use self-attention mechanisms to process sequences in parallel, rep
  doc_01: BERT is a bidirectional encoder pre-trained using masked language modelling and 
  doc_02: The BM25 algorithm ranks documents using term frequency saturation and inverse d
  doc_03: Gradient descent minimizes the loss function by iteratively updating weights in 
  doc_04: Neural networks learn by propagating the error signal backwards through layers u
  doc_05: Proper weight initialization prevents vanishing and exploding gradients, enablin
  doc_06: Batch normalization normalizes layer inputs during training, accelerating conver
  doc_07: Retrieval Augmented Generation grounds language model responses by injecting ret
  doc_08: LoRA fine-tunes large language models by injecting trainable low-rank matrices i
  doc_09: GPTQ quantizes model weights to INT4 post-training by minimizing layer-wise reco
  doc_10: Mixture of Experts models activate only a sparse subs

---

## Part 2 - Hybrid Retrieval (BM25 + SBERT + RRF)

### Reciprocal Rank Fusion Formula

$$\text{RRF}(d) = \frac{1}{k + r_{\text{BM25}}(d)} + \frac{1}{k + r_{\text{SBERT}}(d)}$$

Where $k = 60$ (smoothing constant), $r(d)$ = rank of document $d$ (1-indexed).

The `retrieve()` method returns a dict including both `bm25_rank` and `sbert_rank` for analysis.

In [4]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

class HybridRetriever:
    """
    Combines BM25 (sparse) + SBERT (dense) via Reciprocal Rank Fusion.
    Returns ranked dicts including both retriever ranks for analysis.
    """

    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        doc_vecs = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks  = {int(d): r + 1 for r, d in enumerate(bm25_ranked)}

        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks  = {int(d): r + 1 for r, d in enumerate(sbert_ranked)}

        rrf = {}
        for doc_id in range(len(self.corpus)):
            rrf[doc_id] = (
                1.0 / (self.k + bm25_ranks[doc_id]) +
                1.0 / (self.k + sbert_ranks[doc_id])
            )

        top = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [
            {
                "doc_id":    doc_id,
                "rrf_score": rrf[doc_id],
                "bm25_rank": bm25_ranks[doc_id],
                "sbert_rank": sbert_ranks[doc_id],
                "text":      self.corpus[doc_id],
            }
            for doc_id in top
        ]


hybrid = HybridRetriever(corpus)
print("HybridRetriever built.")
print()

test_queries = [
    "How do neural networks learn?",
    "BM25 term frequency ranking",
    "AdamW optimizer weight decay",
]
for q in test_queries:
    print(f"Query: '{q}'")
    print(f"  {'doc_id':<8} {'RRF score':<12} {'BM25 rank':<12} {'SBERT rank':<12} text")
    print("  " + "-" * 80)
    for r in hybrid.retrieve(q, top_k=3):
        print(f"  doc_{r['doc_id']:<4}  {r['rrf_score']:.6f}   {r['bm25_rank']:<12} {r['sbert_rank']:<12} {r['text'][:55]}")
    print()


HybridRetriever built.

Query: 'How do neural networks learn?'
  doc_id   RRF score    BM25 rank    SBERT rank   text
  --------------------------------------------------------------------------------
  doc_4     0.032787   1            1            Neural networks learn by propagating the error signal b
  doc_10    0.032002   3            2            Mixture of Experts models activate only a sparse subset
  doc_0     0.031054   2            7            Transformers use self-attention mechanisms to process s

Query: 'BM25 term frequency ranking'
  doc_id   RRF score    BM25 rank    SBERT rank   text
  --------------------------------------------------------------------------------
  doc_2     0.032787   1            1            The BM25 algorithm ranks documents using term frequency
  doc_14    0.031258   3            5            Cross-entropy loss measures the divergence between pred
  doc_8     0.030835   8            2            LoRA fine-tunes large language models by injecti

---

## Part 3 - Cross-Encoder Re-Ranking

After Hybrid Retrieval returns the top-K candidates, the **Cross-Encoder** re-scores each `(query, document)` pair jointly. Unlike the bi-encoder (which encodes query and document independently), the cross-encoder reads both together — every query token can attend to every document token.

```
Stage 1: Hybrid Retrieval → top-5 candidates  (fast, recall-focused)
Stage 2: Cross-Encoder    → top-3 re-ranked   (slow, precision-focused)
```

**Note**: The `rerank()` function always accepts the **original user query** (not the HyDE expansion) to preserve relevance signal.

In [5]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded: ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    Re-rank hybrid retrieval candidates using a cross-encoder.

    Args:
        query      : The original user query (not HyDE-expanded).
        candidates : List of dicts from HybridRetriever.retrieve().
        top_k      : Number of top documents to return.

    Returns:
        List of dicts with added 'ce_score' field, sorted descending.
    """
    pairs  = [[query, c["text"]] for c in candidates]
    scores = cross_encoder.predict(pairs)

    for i, c in enumerate(candidates):
        c["ce_score"] = float(scores[i])

    reranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:top_k]


query = "How does attention work in language models?"
candidates = hybrid.retrieve(query, top_k=5)

print(f"\nQuery: '{query}'")
print("\nBefore Re-Ranking (Hybrid order):")
for i, c in enumerate(candidates, 1):
    print(f"  #{i}  [RRF={c['rrf_score']:.5f}] {c['text']}")

reranked = rerank(query, candidates, top_k=3)
print("\nAfter Cross-Encoder Re-Ranking (top 3):")
for i, c in enumerate(reranked, 1):
    print(f"  #{i}  [CE score={c['ce_score']:.4f}] {c['text']}")

Cross-encoder loaded: ms-marco-MiniLM-L-6-v2

Query: 'How does attention work in language models?'

Before Re-Ranking (Hybrid order):
  #1  [RRF=0.03279] LoRA fine-tunes large language models by injecting trainable low-rank matrices into frozen attention weight projections.
  #2  [RRF=0.03175] BERT is a bidirectional encoder pre-trained using masked language modelling and next sentence prediction.
  #3  [RRF=0.03175] Retrieval Augmented Generation grounds language model responses by injecting retrieved documents into the prompt context.
  #4  [RRF=0.03126] Transformers use self-attention mechanisms to process sequences in parallel, replacing recurrent networks entirely.
  #5  [RRF=0.03016] AdamW decouples weight decay from the gradient update in the Adam optimizer, improving regularization in transformer training.

After Cross-Encoder Re-Ranking (top 3):
  #1  [CE score=2.2710] LoRA fine-tunes large language models by injecting trainable low-rank matrices into frozen attention weight p

---

## Part 4 - Query Expansion

**Both** HyDE and Multi-Query are being implemented, giving us two complementary expansion strategies.

### HyDE (Hypothetical Document Embedding)

Ask Groq LLM to generate a *hypothetical ideal answer* and use its embedding as the search vector. The generated text is written in document-style language, so it lands closer to real corpus documents in vector space.

### Multi-Query

Generate 3 paraphrases of the original query, retrieve top-3 for each, then take the **deduplicated union** — covering multiple angles of the same question.

In [9]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

# Option A: HyDE
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Generate a single factual paragraph (3-5 sentences) that would directly answer the following question. Write it as if it were an excerpt from an AI/ML textbook. Be precise and use technical vocabulary."),
    ("human", "{query}")
])
hyde_chain = hyde_prompt | llm | StrOutputParser()

def expand_hyde(query: str) -> str:
    return hyde_chain.invoke({"query": query})

# Option B: Multi-Query
multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an AI assistant helping improve document retrieval.
Given a user question, generate exactly 3 different paraphrases of that question.
Each paraphrase should approach the topic from a slightly different angle.
Return ONLY the 3 questions, one per line. No numbering, no extra text."""),
    ("human", "{query}")
])
multi_query_chain = multi_query_prompt | llm | StrOutputParser()

def expand_multi_query(query: str) -> list[str]:
    raw = multi_query_chain.invoke({"query": query})
    variants = [q.strip() for q in raw.strip().split("\n") if q.strip()][:3]
    return [query] + variants

# Demo
test_q = "How do transformers encode meaning?"
print(f"Original query: '{test_q}'")
print()

hyde_doc = expand_hyde(test_q)
print("[HyDE] Hypothetical Document:")
print(f"  {hyde_doc}")

print()
mq_variants = expand_multi_query(test_q)
print("[Multi-Query] Variants:")
for i, v in enumerate(mq_variants):
    label = "(original)" if i == 0 else f"(variant {i})"
    print(f"  {label}: {v}")

Original query: 'How do transformers encode meaning?'

[HyDE] Hypothetical Document:
  Transformers encode meaning through a self-attention mechanism, which allows the model to weigh the importance of different input elements relative to one another. This is achieved by computing attention weights, which are calculated as the dot product of the query and key vectors, normalized by the square root of the key vector's dimensionality. The resulting attention weights are then used to compute a weighted sum of the value vectors, effectively selecting and combining relevant information from the input sequence to produce a contextualized representation. This process enables the transformer to capture complex relationships and dependencies within the input data, facilitating the encoding of nuanced meaning and context.

[Multi-Query] Variants:
  (original): How do transformers encode meaning?
  (variant 1): What mechanisms do transformers use to represent semantic meaning in text data?
  (vari

---

## Part 5 - End-to-End Pipeline

The full `advanced_rag()` function wires everything together:

```
User Query
    │
    ▼  Step 1: Query Expansion (HyDE — generates hypothetical answer as search vector)
    │
    ▼  Step 2: Hybrid Retrieval (BM25 + SBERT + RRF on expanded query, top-5)
    │
    ▼  Step 3: Cross-Encoder Re-Ranking (original query vs candidates, top-3)
    │
    ▼  Step 4: LLM Generatio (Groq (Llama) — grounded on top-3 re-ranked docs)
    │
    ▼  Final Answer
```

**Design decision**: HyDE expansion is used for retrieval (dense retrieval benefits most from rich query text). The original user query is always passed to the cross-encoder and LLM for correctness.

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

generation_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a knowledgeable AI/ML teaching assistant for a university.
Answer the student's question using ONLY the provided context documents.
If the answer is not in the context, say 'I don't have enough information to answer this.'
Be concise, precise, and educational.

Context:
{context}"""),
    ("human", "{question}")
])

def format_context(docs: list[dict]) -> str:
    return "\n\n".join(f"[Document {i+1}]\n{d['text']}" for i, d in enumerate(docs))

def advanced_rag(user_query: str) -> str:
    """
    Full pipeline: Query Expansion → Hybrid Retrieval → Re-Ranking → LLM Generation.
    Returns the final answer string.
    """
    print(f"Query: '{user_query}'")
    print("=" * 70)

    expanded = expand_hyde(user_query)
    print(f"[1] HyDE Expansion:\n    {expanded[:200]}...")

    candidates = hybrid.retrieve(expanded, top_k=5)
    print(f"\n[2] Hybrid Retrieval (top 5):")
    for i, c in enumerate(candidates, 1):
        print(f"    #{i} [RRF={c['rrf_score']:.5f} | BM25_rank={c['bm25_rank']} | SBERT_rank={c['sbert_rank']}] {c['text'][:70]}")

    top_docs = rerank(user_query, candidates, top_k=3)
    print(f"\n[3] After Re-Ranking (top 3):")
    for i, d in enumerate(top_docs, 1):
        print(f"    #{i} [CE={d['ce_score']:.4f}] {d['text']}")

    context = format_context(top_docs)
    chain   = generation_prompt | llm | StrOutputParser()
    answer  = chain.invoke({"context": context, "question": user_query})

    print(f"\n[4] Final Answer:\n    {answer}")
    print()
    return answer

answer1 = advanced_rag("how do transformers encode meaning?")

Query: 'how do transformers encode meaning?'
[1] HyDE Expansion:
    Transformers encode meaning through a self-attention mechanism, which allows the model to weigh the importance of different input elements relative to one another. This is achieved by computing attent...

[2] Hybrid Retrieval (top 5):
    #1 [RRF=0.03252 | BM25_rank=2 | SBERT_rank=1] Transformers use self-attention mechanisms to process sequences in par
    #2 [RRF=0.03154 | BM25_rank=1 | SBERT_rank=6] Cosine similarity measures the angle between two vectors and is used t
    #3 [RRF=0.03083 | BM25_rank=8 | SBERT_rank=2] AdamW decouples weight decay from the gradient update in the Adam opti
    #4 [RRF=0.03037 | BM25_rank=9 | SBERT_rank=3] LoRA fine-tunes large language models by injecting trainable low-rank 
    #5 [RRF=0.02988 | BM25_rank=5 | SBERT_rank=9] Gradient descent minimizes the loss function by iteratively updating w

[3] After Re-Ranking (top 3):
    #1 [CE=-2.6834] Transformers use self-attention mechanis

In [11]:
answer2 = advanced_rag("optimization techniques for training")

Query: 'optimization techniques for training'
[1] HyDE Expansion:
    During the training process of machine learning models, several optimization techniques can be employed to improve convergence rates and reduce computational costs. One such technique is Stochastic Gr...

[2] Hybrid Retrieval (top 5):
    #1 [RRF=0.03279 | BM25_rank=1 | SBERT_rank=1] Gradient descent minimizes the loss function by iteratively updating w
    #2 [RRF=0.03200 | BM25_rank=3 | SBERT_rank=2] Batch normalization normalizes layer inputs during training, accelerat
    #3 [RRF=0.03105 | BM25_rank=2 | SBERT_rank=7] Dropout randomly deactivates neurons during training to prevent co-ada
    #4 [RRF=0.03102 | BM25_rank=6 | SBERT_rank=3] AdamW decouples weight decay from the gradient update in the Adam opti
    #5 [RRF=0.03033 | BM25_rank=4 | SBERT_rank=8] GPTQ quantizes model weights to INT4 post-training by minimizing layer

[3] After Re-Ranking (top 3):
    #1 [CE=-3.6821] AdamW decouples weight decay from the g

In [12]:
answer3 = advanced_rag("what is the role of the router in mixture of experts?")

Query: 'what is the role of the router in mixture of experts?'
[1] HyDE Expansion:
    In the Mixture of Experts (MoE) architecture, the router plays a crucial role in determining the contribution of each expert network to the final output. The router is typically implemented as a softm...

[2] Hybrid Retrieval (top 5):
    #1 [RRF=0.03279 | BM25_rank=1 | SBERT_rank=1] Mixture of Experts models activate only a sparse subset of expert netw
    #2 [RRF=0.03125 | BM25_rank=4 | SBERT_rank=4] AdamW decouples weight decay from the gradient update in the Adam opti
    #3 [RRF=0.03105 | BM25_rank=7 | SBERT_rank=2] Batch normalization normalizes layer inputs during training, accelerat
    #4 [RRF=0.03105 | BM25_rank=2 | SBERT_rank=7] GPTQ quantizes model weights to INT4 post-training by minimizing layer
    #5 [RRF=0.03054 | BM25_rank=5 | SBERT_rank=6] Gradient descent minimizes the loss function by iteratively updating w

[3] After Re-Ranking (top 3):
    #1 [CE=5.2716] Mixture of Experts mode

---

## Part 6 - Comparison Experiment: Naïve RAG vs Advanced RAG

**Naïve RAG** = Dense-only SBERT cosine retrieval, no expansion, no re-ranking.  
**Advanced RAG** = Full pipeline from Part 5 (HyDE + Hybrid + Cross-Encoder + Groq LLM).

We run the same 3 queries through both pipelines and compare the top retrieved document.

In [14]:
sbert_model = HybridRetriever(corpus).sbert
doc_vecs_naive = sbert_model.encode(corpus, convert_to_numpy=True)
doc_vecs_naive = doc_vecs_naive / np.linalg.norm(doc_vecs_naive, axis=1, keepdims=True)

def naive_rag(user_query: str) -> str:
    q_vec   = sbert_model.encode([user_query], convert_to_numpy=True)[0]
    q_vec   = q_vec / np.linalg.norm(q_vec)
    scores  = doc_vecs_naive @ q_vec
    top_idx = int(np.argsort(scores)[::-1][0])
    return corpus[top_idx]

comparison_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is the role of the router in mixture of experts?",
]

print("Running comparison experiment...")
print()

naive_tops = [naive_rag(q) for q in comparison_queries]
advanced_tops = []

for q in comparison_queries:
    expanded   = expand_hyde(q)
    candidates = hybrid.retrieve(expanded, top_k=5)
    top_docs   = rerank(q, candidates, top_k=3)
    advanced_tops.append(top_docs[0]["text"])

print("Done.")

Running comparison experiment...

Done.


### Comparison Table

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |
|---|---|---|---|
| *"how do transformers encode meaning?"* | Transformers use self-attention mechanisms to process sequences in parallel, replacing recurrent networks entirely. | Transformers use self-attention mechanisms to process sequences in parallel, replacing recurrent networks entirely. | No — both find the same top doc for this semantic query (dense retrieval already works well here) |
| *"optimization techniques for training"* | Gradient descent minimizes the loss function by iteratively updating weights in the direction of the negative gradient. | AdamW decouples weight decay from the gradient update in the Adam optimizer, improving regularization in transformer training. | **Yes** — Advanced RAG surfaces the AdamW doc (a more specific optimization technique) via HyDE expansion + re-ranking |
| *"what is the role of the router in mixture of experts?"* | Mixture of Experts models activate only a sparse subset of expert networks per token, controlled by a learned router. | Mixture of Experts models activate only a sparse subset of expert networks per token, controlled by a learned router. | No — the MoE doc is semantically distinctive enough that both pipelines find it |

### Observations

- **Query 1** (semantic): Naïve RAG performs as well as Advanced RAG because "transformers encode meaning" is semantically close to doc_0 already. Dense retrieval handles this well without expansion.
- **Query 2** (keyword/optimization): Advanced RAG wins — the HyDE expansion generates text mentioning "AdamW", "learning rate", "weight decay", pulling up a more precise document. Naïve RAG stops at the generic gradient descent doc.
- **Query 3** (technical): Both find the correct document. The MoE concept is distinctive enough in vector space that no expansion is needed.

**Key takeaway**: HyDE + Re-Ranking provides the biggest gains on short, ambiguous queries where the user vocabulary differs from document vocabulary — exactly the vocabulary gap this unit is designed to solve.

---

## Bonus Challenge 1 - Weighted RRF

Standard RRF gives equal weight to both retrievers. **Weighted RRF** introduces $\alpha$ to favour one retriever:

$$\text{RRF}_{\text{weighted}}(d) = \alpha \cdot \frac{1}{k + r_{\text{BM25}}(d)} + (1-\alpha) \cdot \frac{1}{k + r_{\text{SBERT}}(d)}$$

We experiment with $\alpha \in \{0.3, 0.5, 0.7\}$ on two contrasting queries:
- **Keyword-heavy**: `"BM25 term frequency ranking"` — BM25 should dominate → higher $\alpha$ better.
- **Semantic**: `"how do neural networks learn from examples?"` — SBERT should dominate → lower $\alpha$ better.

In [15]:
class WeightedHybridRetriever(HybridRetriever):
    """
    Extends HybridRetriever with a weighted RRF formula.
    alpha controls BM25 weight; (1-alpha) controls SBERT weight.
    """

    def retrieve_weighted(self, query: str, alpha: float = 0.5, top_k: int = 5) -> list[dict]:
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks  = {int(d): r + 1 for r, d in enumerate(bm25_ranked)}

        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks  = {int(d): r + 1 for r, d in enumerate(sbert_ranked)}

        rrf = {}
        for doc_id in range(len(self.corpus)):
            rrf[doc_id] = (
                alpha       / (self.k + bm25_ranks[doc_id]) +
                (1 - alpha) / (self.k + sbert_ranks[doc_id])
            )

        top = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [
            {"doc_id": doc_id, "rrf_score": rrf[doc_id],
             "bm25_rank": bm25_ranks[doc_id], "sbert_rank": sbert_ranks[doc_id],
             "text": self.corpus[doc_id]}
            for doc_id in top
        ]


w_hybrid = WeightedHybridRetriever(corpus)

keyword_q  = "BM25 term frequency ranking"
semantic_q = "how do neural networks learn from examples?"

for query in [keyword_q, semantic_q]:
    print(f"Query: '{query}'")
    print(f"  {'alpha':<8} {'Top-1 doc (first 70 chars)'}")
    print("  " + "-" * 75)
    for alpha in [0.3, 0.5, 0.7]:
        result = w_hybrid.retrieve_weighted(query, alpha=alpha, top_k=1)
        print(f"  alpha={alpha}  {result[0]['text'][:70]}")
    print()

print("Interpretation:")
print("  Keyword query  → higher alpha (more BM25 weight) surfaces the exact-match doc faster.")
print("  Semantic query → lower alpha (more SBERT weight) surfaces the semantically closest doc.")

Query: 'BM25 term frequency ranking'
  alpha    Top-1 doc (first 70 chars)
  ---------------------------------------------------------------------------
  alpha=0.3  The BM25 algorithm ranks documents using term frequency saturation and
  alpha=0.5  The BM25 algorithm ranks documents using term frequency saturation and
  alpha=0.7  The BM25 algorithm ranks documents using term frequency saturation and

Query: 'how do neural networks learn from examples?'
  alpha    Top-1 doc (first 70 chars)
  ---------------------------------------------------------------------------
  alpha=0.3  Neural networks learn by propagating the error signal backwards throug
  alpha=0.5  Neural networks learn by propagating the error signal backwards throug
  alpha=0.7  Neural networks learn by propagating the error signal backwards throug

Interpretation:
  Keyword query  → higher alpha (more BM25 weight) surfaces the exact-match doc faster.
  Semantic query → lower alpha (more SBERT weight) surfaces the sema

---

## Bonus Challenge 2 - Chunk Size Study

When documents are long, we must split them into chunks before indexing. Chunk size affects retrieval quality:
- **Small chunks** (50 words): More precise, but may lose sentence context.
- **Large chunks** (200 words): Rich context, but the relevant sentence may be diluted by irrelevant sentences.

We take a 150-word passage about the Transformer architecture, split it at 50, 100, and 200-word boundaries, and measure retrieval quality using cross-encoder scores as a proxy.

In [16]:
long_document = """
The Transformer architecture was introduced in the paper Attention Is All You Need by Vaswani et al. in 2017.
It replaced recurrent neural networks entirely with a mechanism called self-attention.
In self-attention, every token in a sequence computes a weighted sum over all other tokens, allowing the model to capture long-range dependencies without recurrence.
The architecture consists of an encoder and a decoder, each made of stacked layers.
Each encoder layer has two sub-layers: a multi-head self-attention mechanism and a position-wise fully connected feed-forward network.
Residual connections wrap each sub-layer, followed by layer normalization.
The decoder adds a third sub-layer that performs cross-attention over the encoder output.
Positional encodings are added to token embeddings to give the model a sense of token order, since self-attention itself is permutation-invariant.
Multi-head attention runs the attention function in parallel across multiple representation subspaces, then concatenates and projects the results.
The Transformer trained faster and generalized better than previous sequence-to-sequence models on translation benchmarks such as WMT 2014 English-German.
It became the foundation for BERT, GPT, T5, and virtually every subsequent large language model.
""".strip()

def chunk_text(text: str, chunk_size_words: int) -> list[str]:
    words  = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size_words):
        chunk = " ".join(words[i:i + chunk_size_words])
        if chunk:
            chunks.append(chunk)
    return chunks

query = "How does self-attention capture long-range dependencies?"

print(f"Original document: {len(long_document.split())} words")
print(f"Query: '{query}'")
print()

for chunk_size in [50, 100, 200]:
    chunks = chunk_text(long_document, chunk_size)
    pairs  = [[query, c] for c in chunks]
    scores = cross_encoder.predict(pairs)
    best_idx = int(np.argmax(scores))
    best_score = scores[best_idx]

    print(f"Chunk size = {chunk_size} words → {len(chunks)} chunk(s)")
    print(f"  Best CE score: {best_score:.4f}")
    print(f"  Best chunk:    {chunks[best_idx][:120]}...")
    print()

print("Interpretation:")
print("  Smaller chunks isolate the relevant sentence → higher cross-encoder score.")
print("  Larger chunks dilute the signal with unrelated sentences → lower score.")
print("  Optimal chunk size balances precision (small) vs context richness (large).")

Original document: 180 words
Query: 'How does self-attention capture long-range dependencies?'

Chunk size = 50 words → 4 chunk(s)
  Best CE score: 5.7870
  Best chunk:    long-range dependencies without recurrence. The architecture consists of an encoder and a decoder, each made of stacked ...

Chunk size = 100 words → 2 chunk(s)
  Best CE score: 7.5497
  Best chunk:    The Transformer architecture was introduced in the paper Attention Is All You Need by Vaswani et al. in 2017. It replace...

Chunk size = 200 words → 1 chunk(s)
  Best CE score: 6.9631
  Best chunk:    The Transformer architecture was introduced in the paper Attention Is All You Need by Vaswani et al. in 2017. It replace...

Interpretation:
  Smaller chunks isolate the relevant sentence → higher cross-encoder score.
  Larger chunks dilute the signal with unrelated sentences → lower score.
  Optimal chunk size balances precision (small) vs context richness (large).


---

## Bonus Challenge 3 - ColBERT as a Third Retriever in Hybrid RRF

We add **ColBERT-style MaxSim scoring** as a third retriever and fuse three ranked lists with RRF:

$$\text{RRF}_3(d) = \frac{1}{k + r_{\text{BM25}}(d)} + \frac{1}{k + r_{\text{SBERT}}(d)} + \frac{1}{k + r_{\text{ColBERT}}(d)}$$

ColBERT encodes each token independently and scores via:

$$\text{ColBERT}(Q, D) = \sum_{i \in Q} \max_{j \in D} \left( \vec{q}_i \cdot \vec{d}_j \right)$$

This gives token-level late interaction — richer than a single sentence vector, but without full cross-encoder overhead.

In [17]:
class TripleHybridRetriever(HybridRetriever):
    """
    Extends HybridRetriever with ColBERT-style late interaction as a third retriever.
    Fuses BM25 + SBERT + ColBERT with three-way RRF.
    """

    def colbert_score(self, query: str, document: str) -> float:
        q_tokens = query.lower().split()
        d_tokens = document.lower().split()
        q_vecs   = self.sbert.encode(q_tokens, convert_to_numpy=True)
        d_vecs   = self.sbert.encode(d_tokens, convert_to_numpy=True)
        q_vecs   = q_vecs / (np.linalg.norm(q_vecs, axis=1, keepdims=True) + 1e-8)
        d_vecs   = d_vecs / (np.linalg.norm(d_vecs, axis=1, keepdims=True) + 1e-8)
        sim      = q_vecs @ d_vecs.T
        return float(sim.max(axis=1).sum())

    def retrieve_triple(self, query: str, top_k: int = 5) -> list[dict]:
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks  = {int(d): r + 1 for r, d in enumerate(bm25_ranked)}

        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks  = {int(d): r + 1 for r, d in enumerate(sbert_ranked)}

        colbert_scores = [self.colbert_score(query, doc) for doc in self.corpus]
        colbert_ranked = np.argsort(colbert_scores)[::-1]
        colbert_ranks  = {int(d): r + 1 for r, d in enumerate(colbert_ranked)}

        rrf = {}
        for doc_id in range(len(self.corpus)):
            rrf[doc_id] = (
                1.0 / (self.k + bm25_ranks[doc_id]) +
                1.0 / (self.k + sbert_ranks[doc_id]) +
                1.0 / (self.k + colbert_ranks[doc_id])
            )

        top = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [
            {"doc_id": doc_id, "rrf_score": rrf[doc_id],
             "bm25_rank": bm25_ranks[doc_id], "sbert_rank": sbert_ranks[doc_id],
             "colbert_rank": colbert_ranks[doc_id], "text": self.corpus[doc_id]}
            for doc_id in top
        ]


triple = TripleHybridRetriever(corpus)

print("Triple Hybrid (BM25 + SBERT + ColBERT) Retrieval")
print()

for q in ["how does attention capture meaning?", "BM25 term frequency ranking", "AdamW weight decay optimizer"]:
    print(f"Query: '{q}'")
    print(f"  {'doc_id':<8} {'RRF':<12} {'BM25_r':<9} {'SBERT_r':<10} {'ColBERT_r':<12} text")
    print("  " + "-" * 85)
    for r in triple.retrieve_triple(q, top_k=3):
        print(f"  doc_{r['doc_id']:<4}  {r['rrf_score']:.6f}   {r['bm25_rank']:<9} {r['sbert_rank']:<10} {r['colbert_rank']:<12} {r['text'][:45]}")
    print()

print("Observation: ColBERT as a 3rd retriever provides additional signal from token-level")
print("late interaction, particularly helping on queries with rare token overlap.")

Triple Hybrid (BM25 + SBERT + ColBERT) Retrieval

Query: 'how does attention capture meaning?'
  doc_id   RRF          BM25_r    SBERT_r    ColBERT_r    text
  -------------------------------------------------------------------------------------
  doc_8     0.048660   1         3          1            LoRA fine-tunes large language models by inje
  doc_7     0.046708   8         2          3            Retrieval Augmented Generation grounds langua
  doc_12    0.045935   4         5          7            Cosine similarity measures the angle between 

Query: 'BM25 term frequency ranking'
  doc_id   RRF          BM25_r    SBERT_r    ColBERT_r    text
  -------------------------------------------------------------------------------------
  doc_2     0.049180   1         1          1            The BM25 algorithm ranks documents using term
  doc_8     0.046964   8         2          2            LoRA fine-tunes large language models by inje
  doc_14    0.046883   3         5          4     